# 🤖 Taller: Machine Learning para detectar zonas urbanas — **en tu navegador**

**Curso de análisis de imágenes satelitales · Dr. Abel Coronado**

En este taller vas a **entrenar un modelo de aprendizaje automático** que distingue lo urbano de lo no urbano en una imagen satelital real de Aguascalientes — y todo el camino que eso requiere: preparar la imagen, convertir píxeles en objetos, etiquetarlos con datos oficiales de INEGI, y producir un **mapa clasificado por la máquina**. La segmentación, las estadísticas y la red neuronal son las piezas; **el aprendizaje automático es el destino**.

---

## 🧭 Antes de empezar: ¿dónde estás parado?

Esto que ves es **JupyterLab**, el entorno de trabajo estándar de la ciencia de datos. Pero esta versión tiene algo especial: **no está instalada en tu computadora ni corre en un servidor**. Todo el laboratorio — Python, las bibliotecas científicas y geoespaciales, tus datos — vive **dentro de esta pestaña del navegador**, gracias a una tecnología llamada **WebAssembly**.

¿Qué significa eso para ti?

| | |
|---|---|
| 🚫 **Nada que instalar** | No necesitas permisos de administrador ni descargar programas |
| 🔒 **Tus datos no viajan** | El procesamiento ocurre en TU equipo; nada se sube a ninguna nube |
| 🧹 **Cero rastro** | Cierras la pestaña y no queda nada instalado |
| ⚡ **Es Python de verdad** | El mismo código que usarías en un servidor profesional |

**Cómo se usa:** cada bloque gris es una *celda* de código. Haz clic en ella y presiona **Shift + Enter** para ejecutarla. Ve en orden, de arriba hacia abajo. La primera celda tarda unos segundos extra (carga las bibliotecas científicas la primera vez) — es normal.

Empecemos por comprobar que no te estoy mintiendo: 👇

In [ ]:
# ¿Dónde está corriendo este Python? Vamos a preguntárselo directamente.
import sys
import platform

print(f"Versión de Python : {sys.version.split()[0]}")
print(f"Sistema operativo : {sys.platform!r}")
print(f"Arquitectura      : {platform.machine()!r}")
print()
if sys.platform == "emscripten":
    print("✅ 'emscripten' y 'wasm32' significan: este Python corre en WebAssembly,")
    print("   DENTRO de tu navegador. No hay servidor. El laboratorio eres tú. 🚀")
else:
    print("ℹ️ Estás corriendo este cuaderno fuera del navegador (modo local).")

---
## 📚 Teoría 1: la imagen con la que vamos a trabajar

Trabajaremos con un recorte real de la **geomediana Sentinel-2 del año 2020 de Aguascalientes**.

- **Sentinel-2** es una misión de satélites del programa europeo Copernicus que fotografía toda la Tierra cada ~5 días, con píxeles de 10–20 metros. Sus imágenes son **públicas y gratuitas**.
- No mide solo el rojo, verde y azul que ven tus ojos: registra **12 bandas espectrales**, incluyendo infrarrojo cercano (NIR) e infrarrojo de onda corta (SWIR), donde la vegetación, el agua, el suelo y el concreto se distinguen muchísimo mejor.
- Una **geomediana** es el resultado de tomar *todas* las imágenes de un año y calcular, píxel por píxel, un valor mediano robusto. El resultado: una imagen **sin nubes ni sombras**, representativa de todo el año.

Así que cada píxel de nuestra imagen no es un color: es un **vector de 12 números** que describe cómo refleja la luz ese pedacito de 10×10 metros de Aguascalientes. Vamos a verla:

In [ ]:
# Paso 1a — Leer la imagen y conocer sus números
import time
import numpy as np
import rasterio
from rasterio import features
import matplotlib.pyplot as plt
# Estos dos imports también PRE-CARGAN scipy y scikit-learn en el navegador:
# Pyodide instala paquetes al verlos importados en una celda, pero no detecta
# los imports internos de un módulo (como nuestro segmentador shepherd_pure).
import scipy.ndimage
import sklearn.cluster

with rasterio.open("tile_ags_256.tif") as src:
    img = src.read().astype(np.float32)   # (bandas, filas, columnas)
    transform, crs, nodata = src.transform, src.crs, src.nodata

n_bandas, alto, ancho = img.shape
print(f"✓ imagen cargada: {n_bandas} bandas × {alto} filas × {ancho} columnas")
print(f"  cada píxel cubre 10 m × 10 m → el recorte mide ≈ {alto*10/1000:.1f} km por lado")
print(f"  banda 4 (rojo): mín={img[3].min():.0f}  máx={img[3].max():.0f}  media={img[3].mean():.0f}")
print(f"  banda 8 (NIR) : mín={img[7].min():.0f}  máx={img[7].max():.0f}  media={img[7].mean():.0f}")
print()
print("💡 Para la máquina la 'imagen' es esto: un arreglo de números. Nada más — y nada menos.")

In [ ]:
# Paso 1b — Verla como la verían tus ojos (composición en color natural)
# Bandas 4 (rojo), 3 (verde) y 2 (azul); estiramos contraste entre percentiles 2-98
rgb = np.stack([img[3], img[2], img[1]], axis=-1)
p2, p98 = np.percentile(rgb, (2, 98))
rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)

plt.figure(figsize=(6.5, 6.5))
plt.imshow(rgb)
plt.title("Geomediana Sentinel-2 2020 — borde de la ciudad de Aguascalientes")
plt.axis("off")
plt.show()
print("💡 Elegimos a propósito el BORDE de la ciudad: ahí conviven lo urbano y lo agrícola,")
print("   los dos mundos que el modelo deberá aprender a separar.")

---
## 📚 Teoría 2: ¿por qué "segmentar"? Del píxel al objeto

Si quisiéramos clasificar esta imagen píxel por píxel ("¿este píxel es urbano o agrícola?"), tendríamos dos problemas: **ruido** (píxeles aislados mal clasificados, efecto "sal y pimienta") y **falta de contexto** (un píxel gris puede ser una calle, un techo o suelo desnudo — solo, no se sabe).

La alternativa es el enfoque **orientado a objetos (GEOBIA)**: primero agrupamos los píxeles en **segmentos** — regiones contiguas espectralmente homogéneas que corresponden a *cosas* del territorio: una parcela, una manzana, un cuerpo de agua. Después clasificamos los segmentos, no los píxeles.

Usaremos el algoritmo de **Shepherd, Bunting y Dymond (2019)** (*Remote Sensing* 11(6):658), el mismo que usan agencias de monitoreo territorial. Funciona en 3 pasos:

1. **Siembra (K-means):** agrupa los píxeles en ~60 "familias espectrales" según sus 12 bandas — sin importar dónde están.
2. **Aglomerado (clumping):** los píxeles *vecinos* que cayeron en la misma familia se unen en grupos contiguos. Salen miles de grupitos.
3. **Eliminación iterativa:** los grupos demasiado chicos (menos de `minSegmentSize` píxeles) se fusionan, del más pequeño al más grande, con el vecino espectralmente más parecido. Quedan solo objetos de tamaño razonable.

La implementación que vas a ejecutar (`shepherd_pure`) está **validada bit a bit** contra la implementación de referencia (`pyshepseg`): produce exactamente los mismos segmentos.

Veamos el **paso 1** con nuestros propios ojos:

In [ ]:
# PASO 1 — Siembra: K-means agrupa los píxeles en 60 familias espectrales
import shepherd_pure

t0 = time.time()
km = shepherd_pure.fitSpectralClusters(img, numClusters=60, subsamplePcnt=1,
                                       imgNullVal=nodata, fixedKMeansInit=True)
clusters = shepherd_pure.applySpectralClusters(km, img, nodata)
print(f"K-means listo en {time.time()-t0:.1f} s — cada píxel tiene ahora una 'familia' (1 a 60)")

plt.figure(figsize=(12, 5.5))
plt.subplot(1, 2, 1); plt.imshow(rgb); plt.title("La imagen"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(clusters, cmap="tab20", interpolation="nearest")
plt.title("Paso 1: familias espectrales (colores = familias)"); plt.axis("off")
plt.tight_layout(); plt.show()
print("💡 Observa: las familias capturan tipos de cobertura, pero quedan 'salpicadas'.")
print("   Los pasos 2 y 3 convierten esta sal y pimienta en objetos limpios.")

In [ ]:
# PASOS 2 y 3 — Aglomerar y depurar: la segmentación completa
# (reutilizamos el K-means que ya ajustamos: kmeansObj=km)
t0 = time.time()
res = shepherd_pure.doShepherdSegmentation(
    img,
    kmeansObj=km,          # paso 1 ya hecho
    minSegmentSize=50,     # tamaño mínimo de objeto: 50 px = media hectárea
    imgNullVal=nodata)
seg = res.segimg

print(f"Segmentación completa en {time.time()-t0:.1f} s — ¡dentro de tu navegador!")
print(f"  · objetos finales              : {int(seg.max()):,}")
print(f"  · píxeles sueltos absorbidos   : {res.singlePixelsEliminated:,}")
print(f"  · grupitos chicos fusionados   : {res.smallSegmentsEliminated:,}")
print(f"  · umbral espectral de fusión   : {res.maxSpectralDiff:.0f} (calculado automáticamente)")

In [ ]:
# Visualicemos el resultado: las fronteras de los objetos sobre la imagen
from scipy import ndimage

bordes = ndimage.maximum_filter(seg, size=2) != ndimage.minimum_filter(seg, size=2)
vis = rgb.copy()
vis[bordes] = [1, 1, 0]   # fronteras en amarillo

plt.figure(figsize=(7.5, 7.5))
plt.imshow(vis)
plt.title(f"Paso 2+3: {int(seg.max()):,} objetos del territorio (fronteras en amarillo)")
plt.axis("off")
plt.show()
print("💡 Ya no son píxeles: son parcelas, manzanas, caminos. Objetos con sentido.")

---
## 📚 Teoría 3: cada objeto, una fila en una tabla

Para clasificar los objetos necesitamos describirlos con números: sus **estadísticas zonales** — para cada segmento, la media y desviación estándar de cada una de las 12 bandas. Eso convierte la imagen en una **tabla**: una fila por objeto, una columna por característica. Y una tabla ya es territorio conocido: es lo que come cualquier algoritmo de aprendizaje automático.

Como nuestros segmentos están perfectamente alineados al píxel, el cálculo es exacto y rapidísimo con `numpy`:

In [ ]:
import pandas as pd

t0 = time.time()
nseg = int(seg.max()) + 1
flat = seg.ravel()
n_px = np.bincount(flat, minlength=nseg)

tabla = {"segment_id": np.arange(1, nseg), "n_px": n_px[1:]}
for b in range(n_bandas):
    v = img[b].ravel()
    suma  = np.bincount(flat, weights=v,     minlength=nseg)
    suma2 = np.bincount(flat, weights=v * v, minlength=nseg)
    media = np.where(n_px > 0, suma / n_px, 0)
    var   = np.maximum(np.where(n_px > 0, suma2 / n_px, 0) - media**2, 0)
    tabla[f"b{b+1}Mean"]   = media[1:]
    tabla[f"b{b+1}StdDev"] = np.sqrt(var)[1:]

df = pd.DataFrame(tabla)
print(f"Tabla de características en {time.time()-t0:.2f} s: "
      f"{df.shape[0]:,} objetos × {df.shape[1]-2} características espectrales")
df.head()

In [ ]:
# Y de regreso al mapa: poligonizamos los objetos (¡con coordenadas reales!)
# y los pintamos por su reflejo en el infrarrojo cercano (banda 8 ≈ vegetación)
import geopandas as gpd

geoms = ({"properties": {"segment_id": int(v)}, "geometry": g}
         for g, v in features.shapes(seg.astype(np.int32), transform=transform) if v != 0)
gdf = gpd.GeoDataFrame.from_features(geoms, crs=crs).dissolve(by="segment_id", as_index=False)
gdf = gdf.merge(df[["segment_id", "b8Mean"]], on="segment_id")

ax = gdf.plot(column="b8Mean", cmap="RdYlGn", figsize=(7.5, 7.5), linewidth=0, legend=True)
ax.set_title("Objetos coloreados por infrarrojo cercano\n(verde = vegetación vigorosa)")
ax.set_axis_off()
plt.show()
print(f"{len(gdf):,} polígonos georreferenciados — se pueden exportar a GeoPackage y abrir en QGIS")

---
## 📚 Teoría 4: el aprendizaje automático — enseñarle al modelo con ejemplos

Ya tenemos objetos descritos por números. Falta lo importante: que la máquina aprenda a **distinguir lo urbano de lo no urbano**. Para eso necesitamos **ejemplos etiquetados** (la "verdad-terreno"): usaremos los **polígonos de localidades del Marco Geoestadístico de INEGI** — los asentamientos registrados oficialmente. Elegimos a propósito un recorte en el **borde de la ciudad de Aguascalientes**, donde conviven los dos mundos.

La receta clásica del aprendizaje supervisado, que vas a ejecutar completa:

1. **Etiquetar**: a cada objeto le calculamos qué fracción de sus píxeles cae dentro de un asentamiento.
2. **Depurar**: solo entrenamos con objetos *puros* (≥ 90 % de una clase) — ejemplos confiables.
3. **Balancear**: mismo número de ejemplos urbanos y no urbanos, para no sesgar al modelo.
4. **Separar**: 70 % para entrenar, 30 % **que el modelo nunca ve**, para calificarlo honestamente.
5. **Entrenar y evaluar**: usaremos el mismo pipeline del ejercicio completo del curso — una red neuronal (MLP) apilada con un bosque de árboles extra-aleatorios (ExtraTrees).

Todo esto — sí, también el entrenamiento de la red neuronal — ocurre **dentro de tu navegador**.

In [ ]:
# Las etiquetas: ¿qué objetos tocan un asentamiento registrado por INEGI?
with rasterio.open("labels_256.tif") as lsrc:
    etiquetas = lsrc.read(1)        # 1 = asentamiento, 2 = resto del territorio

plt.figure(figsize=(11, 5.2))
plt.subplot(1, 2, 1); plt.imshow(rgb); plt.title("La imagen (borde de la ciudad)"); plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(rgb); plt.imshow(np.where(etiquetas == 1, 1.0, np.nan), cmap="autumn", alpha=0.45)
plt.title("Verdad-terreno: asentamientos INEGI (naranja)"); plt.axis("off")
plt.tight_layout(); plt.show()

# Proporción de etiqueta 'urbano' DENTRO de cada objeto (de nuevo: bincount)
urb_px = np.bincount(flat, weights=(etiquetas == 1).ravel(), minlength=nseg)
df["prop_urbano"] = np.where(n_px > 0, urb_px / n_px, 0)[1:]

print(f"objetos mayormente urbanos   : {(df.prop_urbano >= 0.5).sum()}")
print(f"objetos mayormente no urbanos: {(df.prop_urbano < 0.5).sum()}")

In [ ]:
# Entrenamiento: el MISMO pipeline del ejercicio completo del curso
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

class StackingEstimator(BaseEstimator, TransformerMixin):
    """Apila las predicciones de un modelo como features extra para el siguiente."""
    def __init__(self, estimator):
        self.estimator = estimator
    def fit(self, X, y=None, **kw):
        self.estimator_ = clone(self.estimator); self.estimator_.fit(X, y, **kw); return self
    def transform(self, X):
        X = np.asarray(X); out = [X]
        if hasattr(self.estimator_, "predict_proba"):
            out.append(self.estimator_.predict_proba(X))
        out.append(self.estimator_.predict(X).reshape(-1, 1))
        return np.hstack(out)

# 1) Solo objetos PUROS para entrenar (>=90% de una clase): etiquetas confiables
puros = df[(df.prop_urbano >= 0.9) | (df.prop_urbano <= 0.1)].copy()
puros["clase"] = np.where(puros.prop_urbano >= 0.9, 1, 2)

# 2) Balancear: mismo número de ejemplos de cada clase (muestreo reproducible)
n_min = int(puros.clase.value_counts().min())
balanceado = pd.concat([
    puros[puros.clase == 1].sample(n=n_min, random_state=42),
    puros[puros.clase == 2].sample(n=n_min, random_state=42),
])

feat_cols = [c for c in df.columns if c.startswith("b")]   # 24 features espectrales
X, y = balanceado[feat_cols], balanceado["clase"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

t0 = time.time()
pipeline = make_pipeline(
    StandardScaler(),
    StackingEstimator(MLPClassifier(alpha=0.01, learning_rate_init=0.001,
                                    max_iter=300, random_state=42)),
    ExtraTreesClassifier(bootstrap=False, criterion="entropy", max_features=0.8,
                         n_estimators=100, random_state=42))
pipeline.fit(Xtr, ytr)
acc = accuracy_score(yte, pipeline.predict(Xte))

print(f"Entrenado en {time.time()-t0:.1f} s con {len(Xtr)} objetos ({n_min} por clase)")
print(f"Exactitud en prueba: {acc:.1%}  (sobre {len(Xte)} objetos que el modelo NUNCA vio)")
ConfusionMatrixDisplay(confusion_matrix(yte, pipeline.predict(Xte)),
                       display_labels=["urbano", "no urbano"]).plot(cmap="Blues")
plt.title("Matriz de confusión (conjunto de prueba)"); plt.tight_layout(); plt.show()

In [ ]:
# Y el gran final: clasificar TODOS los objetos y pintar el mapa
df["pred"] = pipeline.predict(df[feat_cols])
print(f"✓ el modelo clasificó los {len(df):,} objetos del recorte (incluidos los ambiguos que nunca vio)")

gdf_ml = gdf.merge(df[["segment_id", "pred"]], on="segment_id")
colores = {1: "#d62728", 2: "#2ca02c"}   # urbano rojo, no-urbano verde
fig, ax = plt.subplots(figsize=(7.5, 7.5))
for clase, nombre in [(1, "urbano"), (2, "no urbano")]:
    gdf_ml[gdf_ml.pred == clase].plot(ax=ax, color=colores[clase], linewidth=0, label=nombre)
ax.legend(); ax.set_axis_off()
ax.set_title("Mapa clasificado por aprendizaje automático\n(entrenado y predicho EN TU NAVEGADOR)")
plt.savefig("mapa_clasificado.png", dpi=130, bbox_inches="tight")
plt.show()

n_u = int((gdf_ml.pred == 1).sum()); n_n = int((gdf_ml.pred == 2).sum())
print(f"mapa clasificado: {n_u} objetos urbanos / {n_n} no urbanos")

---
## 📦 Llévate tus productos

Lo que acabas de producir no es una ilustración: son **productos geoespaciales reales**, georreferenciados, listos para usarse en QGIS o compartirse con tu equipo. La siguiente celda los guarda y te genera **botones de descarga**:

1. `segmentos.tif` — el ráster de objetos (GeoTIFF)
2. `features_urbano.csv` — la tabla de características y predicciones
3. `clasificacion_urbana.gpkg` — el mapa clasificado vectorial
4. `mapa_clasificado.png` — la imagen del mapa para tu presentación

In [ ]:
# 📦 Tus productos, listos para llevar
# Guardamos los 4 productos del taller y generamos botones de descarga.
# Todo se creó en TU equipo; al descargar solo lo mueves a tu carpeta.
import base64
import os
from IPython.display import HTML, display

# 1) ráster de segmentos (GeoTIFF georreferenciado)
with rasterio.open("segmentos.tif", "w", driver="GTiff", height=seg.shape[0],
                   width=seg.shape[1], count=1, dtype="int32",
                   crs=crs, transform=transform) as dst:
    dst.write(seg.astype("int32"), 1)

# 2) tabla de características + predicción (CSV)
df.to_csv("features_urbano.csv", index=False)

# 3) mapa clasificado vectorial (GeoJSON: formato abierto, QGIS lo abre directo)
with open("clasificacion_urbana.geojson", "w") as f:
    f.write(gdf_ml.to_json())

productos = [
    ("segmentos.tif", "Segmentos (GeoTIFF)"),
    ("features_urbano.csv", "Tabla de características (CSV)"),
    ("clasificacion_urbana.geojson", "Mapa clasificado (GeoJSON para QGIS)"),
    ("mapa_clasificado.png", "Mapa (imagen PNG)"),
]

def boton_descarga(path, etiqueta):
    datos = base64.b64encode(open(path, "rb").read()).decode()
    kb = os.path.getsize(path) / 1024
    return (f'<a download="{path}" href="data:application/octet-stream;base64,{datos}" '
            f'style="display:inline-block;margin:5px 10px 5px 0;padding:10px 16px;'
            f'background:#0ea5e9;color:#fff;border-radius:9px;font-weight:600;'
            f'text-decoration:none;font-family:Segoe UI">⬇ {etiqueta} '
            f'<span style="opacity:.75;font-size:12px">({kb:,.0f} KB)</span></a>')

for p, _ in productos:
    print(f"  ✓ guardado: {p}  ({os.path.getsize(p)/1024:,.0f} KB)")
display(HTML("<div>" + "".join(boton_descarga(p, e) for p, e in productos) + "</div>"))
print()
print("💡 El GeoTIFF y el GeoJSON conservan coordenadas reales: ábrelos en QGIS sobre cualquier mapa base.")
print("   (En el laboratorio portable, además, podrás exportar GeoPackage.)")

---
## 🗂️ El taller es tuyo: usa TUS propias imágenes

Hasta aquí trabajaste con nuestro recorte. Pero este taller es **habilitador**: el mismo pipeline funciona con **tu** GeoTIFF multibanda y, si las tienes, **tus** etiquetas.

**Cómo darle tus datos** (dos caminos):

- **Kit offline:** copia tus archivos `.tif` a la carpeta `mis_datos\` (junto al programa del taller) con el Explorador — la celda de abajo los encontrará al instante.
- **Versión en línea:** arrastra tus `.tif` al explorador de archivos del Lab (panel izquierdo, botón ⬆️).

Requisitos de tus datos: imagen **GeoTIFF multibanda** (idealmente sin comprimir o LZW; tamaño sugerido ≤ ~2000×2000 px en navegador — para escenas grandes usa el laboratorio portable); etiquetas opcionales como GeoTIFF de 1 banda **alineado a la imagen** con valores `1` (clase de interés) y `2` (resto).

In [ ]:
# 🗂️ ¿Qué datos tuyos hay disponibles?
import os
mis_archivos = []
origen = ""
try:
    from pyodide.http import pyfetch
    r = await pyfetch("/api/mis_datos")
    if r.ok:
        mis_archivos = [(d["nombre"], d["bytes"]) for d in (await r.json())]
        origen = "carpeta mis_datos (kit offline)"
except Exception:
    pass
if not mis_archivos:
    nuestros = {"tile_ags_256.tif", "labels_256.tif", "segmentos.tif"}
    propios = [f for f in os.listdir(".")
               if f.lower().endswith((".tif", ".tiff")) and f not in nuestros]
    mis_archivos = [(f, os.path.getsize(f)) for f in propios]
    origen = "archivos subidos al explorador del Lab"

if mis_archivos:
    print(f"Encontré {len(mis_archivos)} archivo(s) tuyos en: {origen}\n")
    for n, b in mis_archivos:
        print(f"   📄 {n}   ({b/1e6:,.1f} MB)")
    print("\n👉 Copia el nombre de tu imagen (y etiquetas, si tienes) en la celda de abajo.")
else:
    print("Aún no veo archivos tuyos.")
    print("  · Kit offline: copia tus .tif a la carpeta mis_datos\\ y re-ejecuta esta celda.")
    print("  · En línea: súbelos con el botón ⬆️ del panel izquierdo y re-ejecuta.")

In [ ]:
# 🗂️ TU pipeline: escribe los nombres de TUS archivos y ejecuta
MI_IMAGEN = ""       # ej. "ejemplo_imagen.tif"   (GeoTIFF multibanda)
MIS_ETIQUETAS = ""   # ej. "ejemplo_etiquetas.tif" (opcional: 1=interés, 2=resto)

async def _trae(nombre):
    """Si el archivo está en mis_datos (kit offline), bájalo al entorno."""
    if not os.path.exists(nombre):
        from pyodide.http import pyfetch
        r = await pyfetch("/mis_datos/" + nombre)
        if not r.ok:
            raise FileNotFoundError(f"no encuentro {nombre} (¿está en mis_datos\\ o subido al Lab?)")
        with open(nombre, "wb") as f:
            f.write((await r.bytes()))

if not MI_IMAGEN:
    print("👉 Escribe arriba el nombre de tu imagen (lo viste en la celda anterior) y re-ejecuta.")
else:
    await _trae(MI_IMAGEN)
    with rasterio.open(MI_IMAGEN) as s:
        tu_img = s.read().astype(np.float32)
        tu_tr, tu_crs, tu_nd = s.transform, s.crs, s.nodata
    nb_, al_, an_ = tu_img.shape
    print(f"✓ {MI_IMAGEN}: {nb_} bandas × {al_} × {an_}  (CRS: {tu_crs})")
    if al_ * an_ > 4_200_000:
        print("⚠ imagen grande para el navegador: puede tardar mucho o agotar memoria.")
        print("  Sugerencia: recórtala, o procésala completa en el laboratorio portable.")

    t0 = time.time()
    tu_res = shepherd_pure.doShepherdSegmentation(
        tu_img, numClusters=60, minSegmentSize=50,
        imgNullVal=tu_nd, fixedKMeansInit=True)
    tu_seg = tu_res.segimg
    print(f"✓ segmentada en {time.time()-t0:.1f} s → {int(tu_seg.max()):,} objetos")

    # visualización (RGB si hay ≥3 bandas; si no, la primera banda)
    if nb_ >= 3:
        idx = (3, 2, 1) if nb_ >= 4 else (2, 1, 0)
        tu_rgb = np.stack([tu_img[i] for i in idx], axis=-1)
    else:
        tu_rgb = np.stack([tu_img[0]] * 3, axis=-1)
    q2, q98 = np.percentile(tu_rgb, (2, 98))
    tu_rgb = np.clip((tu_rgb - q2) / max(q98 - q2, 1e-6), 0, 1)
    bb = scipy.ndimage.maximum_filter(tu_seg, 2) != scipy.ndimage.minimum_filter(tu_seg, 2)
    vis_ = tu_rgb.copy(); vis_[bb] = [1, 1, 0]
    plt.figure(figsize=(7, 7)); plt.imshow(vis_)
    plt.title(f"TUS datos, segmentados: {int(tu_seg.max()):,} objetos"); plt.axis("off"); plt.show()

    # features de TUS objetos
    nseg_ = int(tu_seg.max()) + 1
    fl_ = tu_seg.ravel()
    npx_ = np.bincount(fl_, minlength=nseg_)
    tab_ = {"segment_id": np.arange(1, nseg_), "n_px": npx_[1:]}
    for b in range(nb_):
        v = tu_img[b].ravel()
        s1 = np.bincount(fl_, weights=v, minlength=nseg_)
        s2 = np.bincount(fl_, weights=v * v, minlength=nseg_)
        m_ = np.where(npx_ > 0, s1 / npx_, 0)
        tab_[f"b{b+1}Mean"] = m_[1:]
        tab_[f"b{b+1}StdDev"] = np.sqrt(np.maximum(np.where(npx_ > 0, s2 / npx_, 0) - m_**2, 0))[1:]
    tu_df = pd.DataFrame(tab_)
    tu_df.to_csv("mis_features.csv", index=False)
    print(f"✓ tabla de TUS objetos: {tu_df.shape[0]:,} filas × {tu_df.shape[1]-2} features → mis_features.csv")

    if MIS_ETIQUETAS:
        await _trae(MIS_ETIQUETAS)
        with rasterio.open(MIS_ETIQUETAS) as ls:
            tu_lab = ls.read(1)
        u_ = np.bincount(fl_, weights=(tu_lab == 1).ravel(), minlength=nseg_)
        tu_df["prop_interes"] = np.where(npx_ > 0, u_ / npx_, 0)[1:]
        puros_ = tu_df[(tu_df.prop_interes >= 0.9) | (tu_df.prop_interes <= 0.1)].copy()
        puros_["clase"] = np.where(puros_.prop_interes >= 0.9, 1, 2)
        nmin_ = int(puros_.clase.value_counts().min())
        if nmin_ < 10:
            print(f"⚠ pocos ejemplos puros por clase ({nmin_}): el modelo no será confiable.")
        bal_ = pd.concat([puros_[puros_.clase == 1].sample(nmin_, random_state=42),
                          puros_[puros_.clase == 2].sample(nmin_, random_state=42)])
        fc_ = [c for c in tu_df.columns if c.startswith("b")]
        Xtr_, Xte_, ytr_, yte_ = train_test_split(bal_[fc_], bal_["clase"],
                                                  test_size=0.3, random_state=42,
                                                  stratify=bal_["clase"])
        tu_pipe = make_pipeline(
            StandardScaler(),
            StackingEstimator(MLPClassifier(alpha=0.01, learning_rate_init=0.001,
                                            max_iter=300, random_state=42)),
            ExtraTreesClassifier(bootstrap=False, criterion="entropy",
                                 max_features=0.8, n_estimators=100, random_state=42))
        tu_pipe.fit(Xtr_, ytr_)
        acc_ = accuracy_score(yte_, tu_pipe.predict(Xte_))
        print(f"✓ TU modelo entrenado — exactitud en prueba: {acc_:.1%}")
        tu_df["pred"] = tu_pipe.predict(tu_df[fc_])
        pm_ = np.zeros_like(tu_seg, dtype=np.uint8)
        pred_lut = np.zeros(nseg_, dtype=np.uint8)
        pred_lut[tu_df.segment_id.values] = tu_df.pred.values
        pm_ = pred_lut[tu_seg]
        plt.figure(figsize=(7, 7))
        plt.imshow(np.where(pm_ == 1, 1.0, 0.0), cmap="bwr", vmin=0, vmax=1)
        plt.title(f"TU mapa clasificado (rojo = clase de interés) — {acc_:.0%} de exactitud")
        plt.axis("off"); plt.show()
    else:
        print("ℹ sin etiquetas: hicimos segmentación + features. Si agregas MIS_ETIQUETAS, entrenamos TU modelo.")

---
## 🧪 Experimenta tú

La celda de abajo está lista para que juegues con los **dos parámetros clave** del algoritmo. Cambia los valores, ejecútala y observa cómo cambia el mapa:

- `minSegmentSize` — el tamaño mínimo de objeto en píxeles. ¿Qué pasa con 10? ¿Y con 200? *(pista: piensa en qué nivel de detalle territorial necesita tu análisis)*
- `numClusters` — cuántas "familias espectrales" busca el paso 1. ¿Qué pasa con 15? ¿Y con 100?

In [ ]:
# 🧪 Tu laboratorio: cambia estos dos valores y vuelve a ejecutar (Shift+Enter)
MIS_CLUSTERS = 30
MI_TAMANO_MINIMO = 100

t0 = time.time()
mi_res = shepherd_pure.doShepherdSegmentation(
    img, numClusters=MIS_CLUSTERS, minSegmentSize=MI_TAMANO_MINIMO,
    imgNullVal=nodata, fixedKMeansInit=True)
mi_seg = mi_res.segimg

bordes = ndimage.maximum_filter(mi_seg, size=2) != ndimage.minimum_filter(mi_seg, size=2)
vis = rgb.copy(); vis[bordes] = [0, 1, 1]
plt.figure(figsize=(7, 7)); plt.imshow(vis)
plt.title(f"numClusters={MIS_CLUSTERS}, minSegmentSize={MI_TAMANO_MINIMO} → "
          f"{int(mi_seg.max()):,} objetos ({time.time()-t0:.1f} s)")
plt.axis("off"); plt.show()

---
## 🎓 Lo que acabas de lograr

1. Ejecutaste **Python científico completo dentro de tu navegador** — sin instalar nada, sin nube, sin permisos de administrador.
2. Entendiste qué es una **geomediana Sentinel-2** y por qué cada píxel es un vector de 12 mediciones.
3. Aplicaste el algoritmo de **segmentación de Shepherd** — el mismo de uso operativo en agencias de monitoreo — y viste sus 3 pasos por dentro.
4. Convertiste la imagen en una **tabla de objetos con características espectrales**: el insumo exacto para el siguiente paso del curso, la **clasificación con aprendizaje automático** (que también corre aquí, en tu navegador).

En la siguiente sesión: etiquetas de verdad-terreno, entrenamiento de un clasificador y el mapa urbano/no-urbano de Aguascalientes completo.

---
*Implementación de segmentación validada bit a bit contra `pyshepseg` (ubarsc). Datos: geomediana Sentinel-2 2020 (Copernicus / Digital Earth). Plataforma: JupyterLite + Pyodide (WebAssembly). Código y cadena de verificación: [github.com/abxda/portable-satelital](https://github.com/abxda/portable-satelital).*